# 00 · Environment & Data Check

Welcome to the **PySpark FMCG training kit**.

This first notebook just confirms your environment works and the datasets are
present. If every cell runs, you're ready for notebooks 01–04.

**Before running:** make sure you've generated the data:

```bash
uv run python scripts/bootstrap_java.py   # one-time: downloads JDK 17
uv run python generate_data.py            # writes datasets into ./data
```


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "utils" / "spark.py").exists() and _root != _root.parent:
    _root = _root.parent
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils.spark import build_session, read_dim, read_sales_raw, read_sales_typed

spark = build_session("nb00")
print("Spark", spark.version, "ready")

### Dataset overview

Row counts for every generated dataset.

In [ ]:
datasets = {
    "dim_products": read_dim(spark, "dim_products"),
    "dim_stores": read_dim(spark, "dim_stores"),
    "dim_customers": read_dim(spark, "dim_customers"),
    "dim_suppliers": read_dim(spark, "dim_suppliers"),
    "dim_promotions": read_dim(spark, "dim_promotions"),
    "fact_inventory": read_dim(spark, "fact_inventory"),
    "fact_sales (raw csv)": read_sales_raw(spark),
}
for name, df in datasets.items():
    print(f"{name:24s} {df.count():>10,} rows")


### Peek at the data

Inspect the raw (dirty) sales CSV and a clean dimension.

In [ ]:
# The raw sales CSV — every column is a string and the data is deliberately messy.
sales_raw = read_sales_raw(spark)
sales_raw.printSchema()
sales_raw.show(5, truncate=False)

print("A clean reference dimension (Parquet, typed):")
read_dim(spark, "dim_products").printSchema()
read_dim(spark, "dim_products").show(5, truncate=False)


---
🎉 **Solution notebook** — all cells should run top to bottom.